In [0]:
%pip install fitparse

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
def create_hrv_readiness_chart(df_stage, recovery_history_df, stage_num=7):
    """
    Genereert een gecombineerde visualisatie van:
    1. Hartslag (60s gem. in magenta) & Hoogteverloop tijdens Etappe 7 (bovenaan).
    2. Biomarkers HRV & Readiness trend over de dagen met duidelijke herstelverval-accenten.
    """
    plt.style.use('dark_background')
    
    fig = plt.figure(figsize=(12, 9), facecolor='black')
    gs = gridspec.GridSpec(2, 2, height_ratios=[1.3, 1], width_ratios=[1, 1], figure=fig)
    
    # Kleuren
    COLOR_CYAN = '#00F5FF'        # Cyaan
    COLOR_MAGENTA = '#FF2A6D'     # Magenta
    COLOR_ELEV = '#4A5568'        # Donkergrijs
    
    # -------------------------------------------------------------
    # GRAFIEK 1 - Hoogte vs. Hartslag
    # -------------------------------------------------------------
    ax_main = fig.add_subplot(gs[0, :])
    ax_main.set_facecolor('black')
    
    dist_col = None
    for col in ['cum_dist_2d_km', 'distance_km', 'cum_dist_km']:
        if col in df_stage.columns:
            x_dist = df_stage[col]
            dist_col = col
            break
            
    if dist_col is None:
        if 'distance' in df_stage.columns:
            x_dist = df_stage['distance'] / 1000.0
        else:
            x_dist = np.arange(len(df_stage))
    
    elev_col = None
    for col in ['elevation', 'altitude', 'enhanced_altitude', 'altitude_m']:
        if col in df_stage.columns:
            elev_col = col
            break

    ax_elev = ax_main.twinx()
    
    if elev_col:
        min_elev = df_stage[elev_col].min()
        max_elev = df_stage[elev_col].max()
        ax_elev.fill_between(x_dist, df_stage[elev_col], y2=min_elev - 50, color=COLOR_ELEV, alpha=0.35)
        ax_elev.set_ylim(min_elev - 50, max_elev + (max_elev - min_elev) * 0.1)
    else:
        ax_elev.set_visible(False)
        
    ax_elev.set_ylabel('Hoogte (m)', color='#A0A0A0', fontsize=9, fontweight='bold')
    ax_elev.tick_params(colors='#A0A0A0', labelsize=8)
    ax_elev.grid(False)

    if 'heart_rate' in df_stage.columns:
        hr_smooth = df_stage['heart_rate'].rolling(window=60, min_periods=1).mean()
        ax_main.plot(x_dist, hr_smooth, color=COLOR_MAGENTA, lw=2.2, label='60s Gem. Hartslag (BPM)', zorder=4)
        
        avg_hr = df_stage['heart_rate'].mean()
        max_hr = df_stage['heart_rate'].max()
        
        stats_text = (
            f"Gem. Hartslag: {avg_hr:.0f} bpm\n"
            f"Max. Hartslag: {max_hr:.0f} bpm\n"
            f"Herstel-status: Verlaagd)"
        )
        ax_main.text(
            0.02, 0.90, stats_text, 
            transform=ax_main.transAxes, 
            fontsize=9, color='white', fontweight='bold',
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151'),
            zorder=5
        )

    ax_main.set_xlabel('Afstand (km)', color='white', fontsize=10, fontweight='bold')
    ax_main.set_ylabel('Hartslag (bpm)', color=COLOR_MAGENTA, fontsize=10, fontweight='bold')
    ax_main.set_title("ETAPPE PERFORMANCE: FYSIOLOGISCHE STRESS SCORE", color='white', fontsize=11, fontweight='bold', pad=12)
    ax_main.tick_params(colors='white', labelsize=9)
    ax_main.grid(False)
    ax_main.legend(loc='lower right', facecolor='#121212', edgecolor='#333333', labelcolor='white', fontsize=8)

    # Index van Dag 7 bepalen
    hut_idx = recovery_history_df[recovery_history_df['dag'] == f'Dag {stage_num}'].index

    # -------------------------------------------------------------
    # GRAFIEK 2 - HRV Trend over de dagen
    # -------------------------------------------------------------
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.set_facecolor('black')
    
    days = recovery_history_df['dag']
    hrv_values = recovery_history_df['hrv_rmssd']
    
    # Basis HRV lijn
    ax1.plot(days, hrv_values, color=COLOR_CYAN, marker='o', linewidth=2.5, markersize=8, label='HRV Trend')
    
    # Markeer de laagste waarde op Dag 7 in magenta
    if not hut_idx.empty:
        idx = hut_idx[0]
        ax1.scatter(days.iloc[idx], hrv_values.iloc[idx], color=COLOR_MAGENTA, s=120, zorder=5, edgecolor='white', linewidth=1.5, label='Dag 7 (Laagst)')
    
    ax1.set_title('HRV Status', color='white', fontsize=11, fontweight='bold', pad=10)
    ax1.set_ylabel('milliseconde (ms)', color='white', fontsize=9, fontweight='bold')
    ax1.tick_params(colors='white', labelsize=9)
    ax1.grid(False)
    ax1.legend(loc='lower left', facecolor='#121212', edgecolor='#333333', labelcolor='white', fontsize=8)
    
    # -------------------------------------------------------------
    # GRAFIEK 3 - Readiness Score met Kleurverloop
    # -------------------------------------------------------------
    ax2 = fig.add_subplot(gs[1, 1])
    ax2.set_facecolor('black')
    
    readiness = recovery_history_df['readiness_score']
    
    bars = ax2.bar(days, readiness, color=COLOR_CYAN, alpha=1.0, width=0.5, label='Readiness Score')
    
    # Duidelijk accent op Dag 7 (Magenta)
    if not hut_idx.empty:
        idx = hut_idx[0]
        bars[idx].set_color(COLOR_MAGENTA)
        
    ax2.set_title('Training Readiness Score', color='white', fontsize=11, fontweight='bold', pad=10)
    ax2.set_ylabel('Score (0-100)', color='white', fontsize=9, fontweight='bold')
    ax2.set_ylim(0, 105)
    ax2.tick_params(colors='white', labelsize=9)
    ax2.grid(False)
    
    # Waarde labels boven de balken
    for bar in bars:
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f'{int(yval)}', ha='center', va='bottom', color='white', fontsize=8, fontweight='bold')

    plt.tight_layout()
    return fig

In [0]:
# Het FIT-bestand van etappe 7
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260918_stubaier_hohenweg_etappe_7.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)

recovery_data = {
    'dag': ['Dag 2', 'Dag 3', 'Dag 4', 'Dag 5', 'Dag 6', 'Dag 7'],
    'hrv_rmssd': [72, 71, 68, 66, 64, 60],
    'readiness_score': [92, 82, 57, 56, 44, 44]
}
recovery_df = pd.DataFrame(recovery_data)

# 2. Grafiek genereren
fig_etappe7 = create_hrv_readiness_chart(df_clean, recovery_df, stage_num=7)

# 3. Exporteren voor Instagram 
export_for_instagram(fig_etappe7, filename="20260918_hrv_readiness_impact", format_type='landscape')